# CFNA unattended training on Kaggle (leave-it-running journal)

Each **Save & Run All (Commit)** run: pulls the latest checkpoint from the
`cfna-checkpoints` branch of your repo → trains headless for `MINUTES` →
pushes the new checkpoint + metrics back. Rerun it whenever you have GPU
quota (30 h/week free); no babysitting needed.

**One-time setup (Kaggle right sidebar):**
1. Settings → Accelerator → **GPU** (T4 x2 or P100) · Internet → **On**
   (needs a phone-verified account).
2. Add-ons → Secrets → add `GITHUB_TOKEN` = a fine-grained token with
   read/write on `LMMinier/nueronce` (headless runs cannot prompt for it).
3. Click **Save Version → Save & Run All (Commit)** and close the tab.

Gate (PLAN.md): SFT stays off until base held-out **bpb < 1.8** — flip
`RUN_SFT = True` below once a run reports that.

In [ ]:
# 0) Knobs
REPO = "LMMinier/nueronce"
BRANCH = "claude/cfna-neural-core-verify-ldvgn3"   # code branch
CKPT_BRANCH = "cfna-checkpoints"                    # rolling checkpoint snapshot
PRESET = "base_35m"
MINUTES = 540          # ~9 h training inside a 12 h batch session
RUN_SFT = False        # flip to True once held-out bpb < 1.8
SFT_MINUTES = 90

In [ ]:
# 1) Token (Kaggle secret in batch runs; prompt when interactive)
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    from getpass import getpass
    TOKEN = getpass("GitHub token (repo write): ")
AUTH_REPO = f"https://{TOKEN}@github.com/{REPO}"

In [ ]:
# 2) Clone code + install deps
import os
if not os.path.exists("nueronce"):
    !git clone --branch $BRANCH $AUTH_REPO nueronce
%cd nueronce
!git pull
%pip install -q numpy datasets pytest cryptography cffi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — fix Settings→Accelerator")

In [ ]:
# 3) Restore last checkpoint + cumulative metrics from the snapshot branch
import subprocess, glob
os.makedirs("checkpoints", exist_ok=True)
rc = subprocess.run(["git", "clone", "--depth", "1", "--branch", CKPT_BRANCH,
                     AUTH_REPO, "_ckpt_snapshot"], capture_output=True)
if rc.returncode == 0:
    !cp -r _ckpt_snapshot/metrics/. metrics/ 2>/dev/null || true
    for stem in [f"cfna_{PRESET}.pt"] + [f"conv/{n}" for n in ("best.pt", "latest.pt")]:
        parts = sorted(glob.glob(f"_ckpt_snapshot/checkpoints/{stem}.part-*"))
        os.makedirs(os.path.dirname(f"checkpoints/{stem}"), exist_ok=True)
        if parts:
            !cat {' '.join(parts)} > checkpoints/{stem}
        elif os.path.exists(f"_ckpt_snapshot/checkpoints/{stem}"):
            !cp _ckpt_snapshot/checkpoints/{stem} checkpoints/{stem}
    !rm -rf _ckpt_snapshot
    print("restored:", glob.glob("checkpoints/**/*.pt", recursive=True))
else:
    print("no snapshot branch yet — this is run #1, starting fresh")

In [ ]:
# 4) Corpus (re-downloaded each run, ~10 min; sources+licenses in cfna/corpus/stack.py)
!python scripts/dump_corpus_stack.py --out corpus_stack --phase 2 \
    --target-bytes 400000000 --val-every 20
!du -sh corpus_stack

In [ ]:
# 5) AMP safety gate, then BASE PRETRAIN for the session budget (resumable)
!python -m pytest tests/test_gpu_amp.py -q
!python scripts/train_checkpoint.py --preset $PRESET --corpus corpus_stack \
    --minutes $MINUTES --seq 192 --batch 16 --lr 3e-4 --amp --resume \
    --out checkpoints/cfna_$PRESET.pt

In [ ]:
# 6) Gate check + (optional) conversational SFT
import json
hist = json.load(open(f"checkpoints/cfna_{PRESET}.pt.json"))["history"]
best_bpb = min(h["heldout_bpb"] for h in hist)
print(f"best held-out bpb so far: {best_bpb:.3f}  (SFT gate: < 1.8)")
if RUN_SFT and best_bpb < 1.8:
    !python scripts/build_conversation_sft.py --out-dir data/conversation_sft
    !python scripts/train_conversation.py --data data/conversation_sft \
        --preset $PRESET --init-from checkpoints/cfna_$PRESET.pt --loss response \
        --out-dir checkpoints/conv --minutes $SFT_MINUTES --batch 16 --lr 1e-4 --amp --resume
elif RUN_SFT:
    print("RUN_SFT is on but the base has not passed the gate — keep pretraining.")

In [ ]:
# 7) Push snapshot back: checkpoints (split <90MB) + metrics, force-pushed
#    as a rolling branch so history never bloats. Cloud session reads this.
!git config user.email "luisminier79@gmail.com" && git config user.name "LMMinier"
!find checkpoints -name '*.pt' -size +90M -exec sh -c 'split -b 90m "$1" "$1.part-" && rm "$1"' _ {} \;
!git checkout --orphan $CKPT_BRANCH
!git rm -rq --cached . 2>/dev/null || true
!git add -f checkpoints metrics && git commit -m "Checkpoint snapshot (Kaggle run)"
!git push -f $AUTH_REPO $CKPT_BRANCH
!git checkout -f $BRANCH
print("snapshot pushed to branch:", CKPT_BRANCH)

In [ ]:
# 8) (Interactive only) chat with the latest SFT checkpoint
import os
if os.path.exists("checkpoints/conv/best.pt"):
    from cfna.chat import Conversation, load_checkpoint
    model, ck = load_checkpoint("checkpoints/conv/best.pt")
    convo = Conversation(model, system="You are CFNA.", temperature=0.0)
    for q in ["Hello", "What is two plus two?", "What is the capital of France?"]:
        print(f"User: {q}\nCFNA: {convo.say(q)}\n")
else:
    print("no SFT checkpoint yet — base still pretraining toward the bpb gate")